# 08 — MLflow 

**Project:** AI-Powered Energy Consumption Forecasting & Anomaly Detection System



## 8.1 Install the libraries

In [2]:

!pip install -q mlflow xgboost

In [3]:
!pip install mlflow

## 8.2 Imports

In [4]:
import os
import shutil
import numpy as np
import pandas as pd
import mlflow
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

## 8.3 Load the data

These three files were saved by notebook 02:
- `house38_train.csv` — used to train the models
- `house38_val.csv` — used to compare models while tuning
- `house38_test.csv` — used only to report the final result


In [5]:
IN_DIR = "/kaggle/input/notebooks/azizullahasad/02-feature-enginering-and-baseline-model"

train = pd.read_csv(IN_DIR + "/house38_train.csv", index_col=0, parse_dates=True)
val   = pd.read_csv(IN_DIR + "/house38_val.csv",   index_col=0, parse_dates=True)
test  = pd.read_csv(IN_DIR + "/house38_test.csv",  index_col=0, parse_dates=True)

print("Train rows:", len(train))
print("Val rows  :", len(val))
print("Test rows :", len(test))

Train rows: 5756
Val rows  : 823
Test rows : 1645


## 8.4 Feature lists and the reference error

- **1-hour model:** can use the last hour, rolling averages, and calendar features.
- **Day-ahead model:** can only use information that is already known 24 hours earlier.

We also calculate the error of the simple rule *"same hour as yesterday"*. A model is only useful if it beats this rule. The **skill score** tells us by how much:
`skill = 1 - model_MAE / naive_MAE` (above 0 = better than naive, below 0 = worse).

In [6]:
# Features for the 1-hour-ahead model
FEATURES_1H = ["hour", "dow", "is_weekend",
               "lag_1", "lag_2", "lag_24", "lag_48", "lag_168",
               "roll_mean_24", "roll_std_24", "roll_mean_168"]

# Features for the day-ahead model (only information known 24 hours earlier)
FEATURES_24H = ["hour", "dow", "is_weekend", "lag_24", "lag_48", "lag_168"]

# Error of the rule "same hour yesterday" on the test data (used for the skill score)
naive_mae = mean_absolute_error(test["y"], test["lag_24"])
print("Naive 24h MAE:", round(naive_mae, 4), "kW")

Naive 24h MAE: 0.404 kW


## 8.5 Set up MLflow

MLflow will store everything in a local folder called `mlruns`.
An **experiment** is a group of related runs. Ours is called `energy-forecasting-house38`.

In [7]:
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"   
shutil.rmtree("mlruns", ignore_errors=True)      

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("energy-forecasting-house38")

2026/10/21 19:30:10 INFO mlflow.tracking.fluent: Experiment with name 'energy-forecasting-house38' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///kaggle/working/mlruns/285921497093856335', creation_time=1792611010354, effective_trace_archival_retention=None, experiment_id='285921497093856335', last_update_time=1792611010354, lifecycle_stage='active', name='energy-forecasting-house38', tags={}, trace_location=None, workspace='default'>

## 8.6 Helper function: train, score and log one model

To avoid repeating the same code for every model, we write **one function**. It does four steps:
1. Train the model (baselines have no model, so this step is skipped for them).
2. Predict on the validation and test data.
3. Calculate MAE, RMSE and the skill score.
4. Write the parameters and metrics to MLflow.


In [8]:
def run_model(run_name, horizon, features, base_lag, model=None, params={}):

    # One "run" = one row in the MLflow table
    with mlflow.start_run(run_name=run_name + " " + horizon):

        # Step 1: train (baselines have model=None, so nothing to train)
        if model is not None:
            model.fit(train[features], train["y"] - train[base_lag])

        # Step 2: predict (prediction = base lag + predicted change)
        if model is not None:
            val_pred  = val[base_lag]  + model.predict(val[features])
            test_pred = test[base_lag] + model.predict(test[features])
        else:
            val_pred  = val[base_lag]
            test_pred = test[base_lag]

        # Step 3: calculate the errors
        val_mae   = mean_absolute_error(val["y"], val_pred)
        test_mae  = mean_absolute_error(test["y"], test_pred)
        test_rmse = np.sqrt(mean_squared_error(test["y"], test_pred))
        skill     = 1 - test_mae / naive_mae

        # Step 4: log parameters and metrics to MLflow
        mlflow.log_param("model", run_name)
        mlflow.log_param("horizon", horizon)
        for name, value in params.items():
            mlflow.log_param(name, value)

        mlflow.log_metric("val_mae", val_mae)
        mlflow.log_metric("test_mae", test_mae)
        mlflow.log_metric("test_rmse", test_rmse)
        mlflow.log_metric("skill_vs_naive24", skill)

        print(run_name, horizon, "-> test MAE:", round(test_mae, 4), "kW")

## 8.7 Models for 1 hour ahead

Three runs:
- **Persistence** — baseline: next hour = last hour.
- **Random Forest** — the model we used in the project.
- **XGBoost** — a second machine learning model to compare against.

In [9]:
# Baseline: the next hour equals the last hour
run_model("Persistence", "1h", FEATURES_1H, "lag_1")

# Random Forest (same settings as notebook 02)
rf_1h = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.5,
                              n_jobs=-1, random_state=42)
run_model("RandomForest", "1h", FEATURES_1H, "lag_1", rf_1h,
          {"n_estimators": 300, "min_samples_leaf": 3})

# XGBoost
xgb_1h = XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05,
                      subsample=0.8, random_state=42)
run_model("XGBoost", "1h", FEATURES_1H, "lag_1", xgb_1h,
          {"n_estimators": 400, "max_depth": 5, "learning_rate": 0.05})

Persistence 1h -> test MAE: 0.2691 kW
RandomForest 1h -> test MAE: 0.2892 kW
XGBoost 1h -> test MAE: 0.3095 kW


## 8.8 Models for day ahead (24 hours)

The same three runs, but with the day-ahead features.
The baseline here is **seasonal naive**: the value is the same as the same hour yesterday.

In [10]:
# Baseline: same hour yesterday
run_model("SeasonalNaive", "24h", FEATURES_24H, "lag_24")

# Random Forest
rf_24h = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.5,
                               n_jobs=-1, random_state=42)
run_model("RandomForest", "24h", FEATURES_24H, "lag_24", rf_24h,
          {"n_estimators": 300, "min_samples_leaf": 3})

# XGBoost
xgb_24h = XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05,
                       subsample=0.8, random_state=42)
run_model("XGBoost", "24h", FEATURES_24H, "lag_24", xgb_24h,
          {"n_estimators": 400, "max_depth": 5, "learning_rate": 0.05})

SeasonalNaive 24h -> test MAE: 0.404 kW
RandomForest 24h -> test MAE: 0.3863 kW
XGBoost 24h -> test MAE: 0.4199 kW


## 8.9 LSTM (from notebook 03)

The LSTM was trained in notebook 03, so we do not train it again here. We only **write its results to MLflow** so it appears in the same table.

**Before running:** replace the two `0.0` values with the real numbers printed in notebook 03.
Do not run this cell with `0.0`, otherwise the table will show a wrong result.

In [11]:
LSTM_TEST_MAE = None     # put the real test MAE from notebook 03 here
LSTM_TEST_RMSE = None    # put the real test RMSE from notebook 03 here

if LSTM_TEST_MAE is None or LSTM_TEST_RMSE is None:
    print("Skipped: fill in the two numbers above first")
else:
    with mlflow.start_run(run_name="LSTM 1h"):
        mlflow.log_param("model", "LSTM")
        mlflow.log_param("horizon", "1h")
        mlflow.log_metric("test_mae", LSTM_TEST_MAE)
        mlflow.log_metric("test_rmse", LSTM_TEST_RMSE)

Skipped: fill in the two numbers above first


## 8.10 Log the deployed models

These are the final files saved by notebook 06 and used by the API:
- `energy_forecast_1h.pkl`
- `energy_forecast_24h.pkl`
- `anomaly_model.pkl`

We store them as MLflow **artifacts**, together with the anomaly settings, so we always know which files belong to the deployed version.

**Change `PKL_DIR` to the folder where these three files are.**

In [12]:
import joblib

# Folder that contains the three .pkl files from notebook 06 (change if needed)
PKL_DIR = "/kaggle/input/notebooks/azizullahasad/06-model-export-for-deployment"

with mlflow.start_run(run_name="deployed models"):

    # Anomaly settings are read from the saved model, so they always match the deployed file
    anomaly_path = os.path.join(PKL_DIR, "anomaly_model.pkl")
    if os.path.exists(anomaly_path):
        anomaly = joblib.load(anomaly_path)
        mlflow.log_param("anomaly_k", anomaly["k_default"])
        mlflow.log_param("anomaly_sigma", round(anomaly["sigma"], 4))
        mlflow.log_param("anomaly_median", round(anomaly["median"], 4))

    for file_name in ["energy_forecast_1h.pkl", "energy_forecast_24h.pkl", "anomaly_model.pkl"]:
        path = os.path.join(PKL_DIR, file_name)
        if os.path.exists(path):
            mlflow.log_artifact(path)
            print("Logged:", file_name)
        else:
            print("NOT FOUND (check PKL_DIR):", file_name)

Logged: energy_forecast_1h.pkl
Logged: energy_forecast_24h.pkl
Logged: anomaly_model.pkl


## 8.11 Compare all runs in one table

This table is the main result of this notebook. It is sorted by test MAE, so the best model is at the top.
Take a screenshot of it for the README.

In [13]:
# Read all runs from MLflow
runs = mlflow.search_runs()

# Keep only the useful columns and give them short names
table = runs[["tags.mlflow.runName", "params.horizon",
              "metrics.val_mae", "metrics.test_mae",
              "metrics.test_rmse", "metrics.skill_vs_naive24"]]
table.columns = ["run", "horizon", "val_mae", "test_mae", "test_rmse", "skill_vs_naive24"]

# Remove runs without a test MAE (for example the "deployed models" run), then sort
table = table.dropna(subset=["test_mae"]).sort_values(["horizon", "test_mae"])
table.round(4).reset_index(drop=True)

,run,horizon,val_mae,test_mae,test_rmse,skill_vs_naive24
0,Persistence 1h,1h,0.5869,0.2691,0.5043,0.3340
1,RandomForest 1h,1h,0.5122,0.2892,0.4945,0.2841
2,XGBoost 1h,1h,0.5579,0.3095,0.5379,0.2339
3,RandomForest 24h,24h,0.5731,0.3863,0.6326,0.0439
4,SeasonalNaive 24h,24h,0.6585,0.4040,0.6875,0.0000
5,XGBoost 24h,24h,0.6113,0.4199,0.6912,-0.0392


## 8.12 Save the MLflow records for download

Kaggle cannot show the MLflow web page easily. So we zip the `mlruns` folder, download it from the **Output** panel, and open the web page on our own computer.

In [14]:
shutil.make_archive("mlruns_backup", "zip", "mlruns")
print("Saved: mlruns_backup.zip")

Saved: mlruns_backup.zip
